##  01. Introduction

# Upcoming Movie Predictions

## Objective
Use the final saved models to predict worldwide box office revenue
and blockbuster probabilities for the selected upcoming movies.

## Workflow
1. Locate saved models and prediction data.
2. Load models and their saved configuration.
3. Validate prediction features against model requirements.
4. Generate revenue predictions and blockbuster probabilities.
5. Review limitations and export results.

## Rules
- Use pre-release information only.
- Preserve the training feature definitions and preprocessing.
- Use the classification threshold selected during validation.
- Do not retrain models or adjust thresholds in this notebook.

## 02. Import libraries

In [8]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


## 03. Project Folders

In [9]:
# Find the project root from the notebook's working directory.
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in [current_dir, *current_dir.parents]
        if (folder / "Data" / "Models" / "Final").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate Data/Models/Final. "
        "Open this notebook inside your Movie Analysis Predictor project."
    )

MODEL_DIR = PROJECT_ROOT / "Data" / "Models" / "Final"
EVALUATION_DIR = PROJECT_ROOT / "Data" / "Results" / "Final_Evaluation"
FEATURE_PATH = (
    PROJECT_ROOT / "Data" / "Features" / "prediction_target_features.csv"
)

print(f"Project root: {PROJECT_ROOT}")

file_records = []

for category, folder in [
    ("Saved model / configuration", MODEL_DIR),
    ("Final evaluation", EVALUATION_DIR),
]:
    if folder.is_dir():
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                file_records.append({
                    "category": category,
                    "file": str(file.relative_to(PROJECT_ROOT)),
                    "size_kb": round(file.stat().st_size / 1024, 2),
                })

display(pd.DataFrame(
    file_records,
    columns=["category", "file", "size_kb"],
))

if not FEATURE_PATH.is_file():
    raise FileNotFoundError(
        f"Prediction feature dataset not found: {FEATURE_PATH}"
    )

print(f"\nPrediction features: {FEATURE_PATH.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor


,category,file,size_kb
0,Saved model / configuration,Data\Models\Final\blockbuster_classifier.joblib,21139.99
1,Saved model / configuration,Data\Models\Final\model_manifest.json,1.59
2,Saved model / configuration,Data\Models\Final\preprocessor.joblib,5.54
3,Saved model / configuration,Data\Models\Final\revenue_regressor.joblib,899.65
4,Final evaluation,Data\Results\Final_Evaluation\classification_confusion_matrix.csv,0.10
5,Final evaluation,Data\Results\Final_Evaluation\classification_test_metrics.json,0.17
6,Final evaluation,Data\Results\Final_Evaluation\regression_test_metrics.json,0.18
7,Final evaluation,Data\Results\Final_Evaluation\test_predictions.csv,22.28



Prediction features: Data\Features\prediction_target_features.csv


## 04. Loading and inspecting prediction features

In [10]:
prediction_features = pd.read_csv(FEATURE_PATH)

if prediction_features.empty:
    raise ValueError("The prediction feature dataset contains no movies.")

print(f"Movies: {len(prediction_features):,}")
print(f"Columns: {prediction_features.shape[1]:,}")

display(prediction_features.head(10))

feature_audit = pd.DataFrame({
    "column": prediction_features.columns,
    "dtype": prediction_features.dtypes.astype(str).to_numpy(),
    "missing_count": prediction_features.isna().sum().to_numpy(),
    "missing_percent": (
        prediction_features.isna().mean().mul(100).round(2).to_numpy()
    ),
})

display(feature_audit)

print(
    "\nInspection complete. Missing values have been preserved "
    "for validation against the saved preprocessing."
)

Movies: 19
Columns: 18


,title,cohort,release_date,log_budget,release_year,release_month,release_quarter,holiday_release,director_prior_films,director_prior_avg_gross,star_prior_films,star_prior_avg_gross,company_prior_films,company_prior_avg_gross,primary_genre,release_season,worldwide_box_office,log_worldwide_box_office
0,Spider-Man: Brand New Day,released_backtest,2026-07-31,19.231611,2026,7,3,0,2,1.186685e+07,3,7.180152e+08,332,1.295450e+08,Action,summer,2.402937e+09,21.599958
1,Michael,released_backtest,2026-04-24,18.951309,2026,4,2,0,12,1.148616e+08,0,NaN,4,8.660476e+07,Biography,spring,1.021427e+09,20.744466
2,The Odyssey,released_backtest,2026-07-17,19.336971,2026,7,3,0,11,4.685044e+08,25,1.616126e+08,1,5.270163e+08,Adventure,summer,1.627387e+09,21.210241
3,Scary Movie,released_backtest,2026-06-05,17.216708,2026,6,2,0,3,3.591527e+07,8,1.452784e+08,74,5.097942e+07,Comedy,summer,2.312778e+08,19.259130
4,Mortal Kombat II,released_backtest,2026-05-08,18.197537,2026,5,2,0,0,NaN,4,3.527871e+07,174,1.142747e+08,Action,summer,1.294701e+08,18.678961
5,The Super Mario Galaxy Movie,released_backtest,2026-04-03,18.515991,2026,4,2,0,0,NaN,6,8.797427e+08,2,1.097122e+09,Animation,spring,1.012793e+09,20.735978
6,The Devil Wears Prada 2,released_backtest,2026-05-01,18.420681,2026,5,2,0,7,1.156514e+08,25,9.141235e+07,0,NaN,Comedy,summer,6.928175e+08,20.356277
7,Toy Story 5,released_backtest,2026-06-19,18.980297,2026,6,2,0,4,6.935936e+08,41,2.427420e+08,12,6.571954e+08,Animation,summer,1.137320e+09,20.851940
8,GOAT,released_backtest,2026-02-13,18.197537,2026,2,1,0,0,NaN,0,NaN,4,2.319894e+08,Animation,winter,1.953569e+08,19.090339
9,Backrooms,released_backtest,2026-05-29,16.118096,2026,5,2,0,0,NaN,6,4.158317e+07,8,2.547500e+07,Horror,summer,3.931673e+08,19.789746


,column,dtype,missing_count,missing_percent
0,title,object,0,0.00
1,cohort,object,0,0.00
2,release_date,object,0,0.00
3,log_budget,float64,9,47.37
4,release_year,int64,0,0.00
5,release_month,int64,0,0.00
6,release_quarter,int64,0,0.00
7,holiday_release,int64,0,0.00
8,director_prior_films,int64,0,0.00
9,director_prior_avg_gross,float64,9,47.37



Inspection complete. Missing values have been preserved for validation against the saved preprocessing.


## 05. LOAD SAVED MODEL FILES

In [11]:
from sklearn.base import is_classifier, is_regressor

model_files = sorted(
    path
    for path in MODEL_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".joblib", ".pkl", ".pickle"}
)

if not model_files:
    raise FileNotFoundError(f"No saved model files found in {MODEL_DIR}")

loaded_objects = {}
model_inventory = []

for path in model_files:
    # These must be your own trusted model files.
    saved_object = joblib.load(path)
    relative_name = str(path.relative_to(MODEL_DIR))
    loaded_objects[relative_name] = saved_object

    # Support models saved directly or inside a dictionary.
    candidates = (
        saved_object.items()
        if isinstance(saved_object, dict)
        else [("model", saved_object)]
    )

    for object_name, candidate in candidates:
        if not callable(getattr(candidate, "predict", None)):
            continue

        model_kind = (
            "classification" if is_classifier(candidate)
            else "regression" if is_regressor(candidate)
            else "unknown"
        )

        model_inventory.append({
            "file": relative_name,
            "object_name": object_name,
            "model_kind": model_kind,
            "python_type": type(candidate).__name__,
        })

model_inventory_df = pd.DataFrame(model_inventory)

if model_inventory_df.empty:
    raise ValueError("Files loaded, but no prediction models were identified.")

display(model_inventory_df)

,file,object_name,model_kind,python_type
0,blockbuster_classifier.joblib,model,classification,RandomForestClassifier
1,revenue_regressor.joblib,model,regression,HistGradientBoostingRegressor


## 06. SELECT THE FINAL REGRESSION AND CLASSIFICATION MODELS

In [12]:
def select_saved_model(model_kind):
    matches = model_inventory_df.loc[
        model_inventory_df["model_kind"].eq(model_kind)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one {model_kind} model, found {len(matches)}. "
            "Review the Cell 5 inventory before continuing."
        )

    selected = matches.iloc[0]
    saved_object = loaded_objects[selected["file"]]

    model = (
        saved_object[selected["object_name"]]
        if isinstance(saved_object, dict)
        else saved_object
    )

    print(f"{model_kind.capitalize()}: {selected['file']}")
    return model


regression_model = select_saved_model("regression")
classification_model = select_saved_model("classification")

print("\nBoth final models loaded successfully.")

Regression: revenue_regressor.joblib
Classification: blockbuster_classifier.joblib

Both final models loaded successfully.


## 07. VALIDATE AND ORDER THE PREDICTION FEATURES

In [13]:
def prepare_prediction_inputs(model, data, model_label):
    expected_columns = getattr(model, "feature_names_in_", None)

    if expected_columns is None:
        raise ValueError(
            f"{model_label} does not expose its original input column names. "
            "We need the saved training feature list and preprocessing "
            "before making predictions."
        )

    expected_columns = list(expected_columns)

    if data.columns.duplicated().any():
        raise ValueError("The prediction dataset contains duplicate columns.")

    missing_columns = [
        column for column in expected_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{model_label} requires missing columns: {missing_columns}. "
            "Recreate them using the training feature definitions."
        )

    model_inputs = data.loc[:, expected_columns].copy()

    numeric_inputs = model_inputs.select_dtypes(include=[np.number])
    infinite_columns = numeric_inputs.columns[
        np.isinf(numeric_inputs).any(axis=0)
    ].tolist()

    if infinite_columns:
        raise ValueError(
            f"{model_label} has infinite values in: {infinite_columns}"
        )

    print(
        f"{model_label}: {len(model_inputs):,} movies, "
        f"{len(expected_columns)} input features."
    )

    display(pd.DataFrame({
        "feature": expected_columns,
        "dtype": model_inputs.dtypes.astype(str).to_numpy(),
        "missing_count": model_inputs.isna().sum().to_numpy(),
    }))

    return model_inputs


X_prediction_regression = prepare_prediction_inputs(
    regression_model,
    prediction_features,
    "Revenue model",
)

X_prediction_classification = prepare_prediction_inputs(
    classification_model,
    prediction_features,
    "Blockbuster model",
)

print(
    "Column checks passed. Before prediction, confirm that the saved "
    "models include the training preprocessing and verify how the "
    "revenue target was transformed."
)

ValueError: Revenue model requires missing columns: ['director_has_prior_history', 'star_has_prior_history', 'company_has_prior_history', 'budget_missing', 'primary_genre_Action', 'primary_genre_Adventure', 'primary_genre_Animation', 'primary_genre_Biography', 'primary_genre_Comedy', 'primary_genre_Crime', 'primary_genre_Drama', 'primary_genre_Family', 'primary_genre_Fantasy', 'primary_genre_Horror', 'primary_genre_Music', 'primary_genre_Mystery', 'primary_genre_Romance', 'primary_genre_Sci-Fi', 'primary_genre_Thriller', 'primary_genre_Western', 'release_season_Unknown', 'release_season_fall', 'release_season_holiday', 'release_season_spring', 'release_season_summer', 'release_season_winter']. Recreate them using the training feature definitions.